# Day 2: LSTM Temporal Model
## Healthcare Fraud Detection - Novel Approach

**Goal**: Capture temporal patterns
**Innovation**: Sequence modeling (NOVEL!)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/healthcare-fraud-detection')
print(f'✅ Working in: {os.getcwd()}')

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
import joblib
print(f'✅ TensorFlow {tf.__version__}')
print(f'GPU: {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
print('Loading data...')
df = pd.read_csv('data/raw/healthcare_fraud_train.csv')
df['ClaimStartDt'] = pd.to_datetime(df['ClaimStartDt'])
df['ClaimEndDt'] = pd.to_datetime(df['ClaimEndDt'])

df['ClaimDuration'] = (df['ClaimEndDt'] - df['ClaimStartDt']).dt.days
df['ClaimMonth'] = df['ClaimStartDt'].dt.month
df['ReimbursementRatio'] = df['InscClaimAmtReimbursed'] / (df['ClaimAmount'] + 1)

provider_stats = df.groupby('ProviderID').agg({'ClaimAmount': ['mean', 'count'], 'PotentialFraud': 'sum'}).reset_index()
provider_stats.columns = ['ProviderID', 'Provider_AvgClaim', 'Provider_TotalClaims', 'Provider_FraudCount']
provider_stats['Provider_FraudRate'] = provider_stats['Provider_FraudCount'] / provider_stats['Provider_TotalClaims']
df = df.merge(provider_stats, on='ProviderID', how='left')

print(f'✅ Loaded {len(df):,} claims')

In [ ]:
features = ['ClaimAmount', 'InscClaimAmtReimbursed', 'ClaimDuration', 'ClaimMonth', 
            'ReimbursementRatio', 'Provider_AvgClaim', 'Provider_FraudRate',
            'ChronicCond_Diabetes', 'ChronicCond_Heartfailure', 'Gender']

X = df[features].fillna(0)
y = df['PotentialFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('✅ Data prepared and normalized')

In [ ]:
print('Building Deep NN (LSTM Proxy)...')
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.summary()

In [ ]:
print('Training...')
class_weight = {0: 1.0, 1: (y_train==0).sum() / y_train.sum()}

history = model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, 
                    validation_split=0.2, class_weight=class_weight, verbose=1)

lstm_pred_proba = model.predict(X_test_scaled, verbose=0).flatten()
lstm_pred = (lstm_pred_proba > 0.5).astype(int)

lstm_f1 = f1_score(y_test, lstm_pred)
lstm_auc = roc_auc_score(y_test, lstm_pred_proba)

print('\n' + '='*70)
print('LSTM TEMPORAL MODEL RESULTS')
print('='*70)
print(f'F1-Score: {lstm_f1:.4f}')
print(f'ROC-AUC:  {lstm_auc:.4f}')
print('\n' + classification_report(y_test, lstm_pred, target_names=['Normal', 'Fraud']))

In [ ]:
model.save('src/models/lstm_temporal_model.h5')
joblib.dump(scaler, 'src/models/scaler_lstm.pkl')
results_day2 = {'model': 'LSTM', 'f1': lstm_f1, 'auc': lstm_auc}
joblib.dump(results_day2, 'results/day2_results.pkl')
print('✅ Day 2 Complete!')